
# Rookie Projection Models

This notebook builds statistical comps for incoming rookies based on college production, athletic testing, and historical rookie transition rates. It also incorporates pre-season usage patterns from training camp and exhibition games to produce an initial Boom/Bust tiering for the draft kit.


In [ ]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
np.random.seed(42)


## Create Historical Player Data

In [ ]:

positions = ['QB', 'RB', 'WR', 'TE']
hist_players = []
for pos in positions:
    for i in range(50):
        hist_players.append({
            'player': f'{pos}_player_{i}',
            'position': pos,
            'college_yards': np.random.normal(3000 if pos=='QB' else 1200 if pos=='RB' else 1000 if pos=='WR' else 800, 200),
            'college_tds': np.random.normal(30 if pos=='QB' else 12, 5),
            'forty_time': np.random.normal(4.7 if pos=='QB' else 4.5 if pos in ['RB','WR'] else 4.75, 0.1),
            'bench_press': np.random.normal(15 if pos=='QB' else 20,5),
            'rookie_fantasy_points': np.random.normal(250 if pos=='QB' else 200 if pos=='RB' else 180, 50)
        })

ehist_df = pd.DataFrame(hist_players)


## Historical Rookie Transition Rates

In [ ]:

transition_rates = ehist_df.groupby('position')['rookie_fantasy_points']    .apply(lambda x: (x > 200).mean())    .reset_index(name='p_above_200')
transition_rates


## Incoming Rookie Data

In [ ]:

rookies = []
for pos in positions:
    for i in range(5):
        rookies.append({
            'player': f'{pos}_rookie_{i}',
            'position': pos,
            'college_yards': np.random.normal(3200 if pos=='QB' else 1300 if pos=='RB' else 1100 if pos=='WR' else 900, 150),
            'college_tds': np.random.normal(32 if pos=='QB' else 14, 4),
            'forty_time': np.random.normal(4.68 if pos=='QB' else 4.48 if pos in ['RB','WR'] else 4.72, 0.08),
            'bench_press': np.random.normal(16 if pos=='QB' else 22,4),
            'preseason_usage': np.random.uniform(0.2, 0.8)
        })
rookie_df = pd.DataFrame(rookies)
rookie_df.head()


## Find Statistical Comparables

In [ ]:

features = ['college_yards','college_tds','forty_time','bench_press']
X_hist = ehist_df[features]
X_rook = rookie_df[features]
scaler = StandardScaler()
X_hist_scaled = scaler.fit_transform(X_hist)
X_rook_scaled = scaler.transform(X_rook)

nn = NearestNeighbors(n_neighbors=5, metric='euclidean')
_ = nn.fit(X_hist_scaled)

# Get indices of nearest neighbors for each rookie
distances, indices = nn.kneighbors(X_rook_scaled)

comp_names = ehist_df.loc[indices.flatten(), 'player'].values.reshape(len(rookie_df), -1)
rookie_df['comps'] = [list(c) for c in comp_names]
rookie_df[['player','position','comps']].head()


## Integrate Transition Rates and Preseason Usage

In [ ]:

rookie_df = rookie_df.merge(transition_rates, on='position', how='left')

# compute expected points from comps and boom/bust scores
rookie_df['mean_comp_points'] = [ehist_df.loc[indices[i], 'rookie_fantasy_points'].mean() for i in range(len(rookie_df))]
rookie_df['boom_score'] = rookie_df['mean_comp_points'] * (0.5 + rookie_df['preseason_usage'])
rookie_df['bust_score'] = rookie_df['mean_comp_points'] * (1 - rookie_df['preseason_usage'])
conditions = [
    rookie_df['boom_score'] > rookie_df['mean_comp_points']*1.1,
    rookie_df['bust_score'] < rookie_df['mean_comp_points']*0.9
]
choices = ['Boom','Bust']
rookie_df['tier'] = np.select(conditions, choices, default='Neutral')
rookie_df[['player','position','preseason_usage','p_above_200','tier']]


## Summary

In [ ]:
rookie_df[['player','position','tier','mean_comp_points','boom_score','bust_score']]